<a href="https://colab.research.google.com/github/ntatfff/todo/blob/202411241258/ColabRadiomicsFeatureExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
kernel = 1
className = 'firstorder'
typeOfVoxel = 'tumor'

In [2]:
# Parameters
kernel = 5
className = "gldm"
typeOfVoxel = "non_tumor"


In [3]:
# Utilities: loaders and validation
import os
from typing import List, Tuple
import numpy as np
import SimpleITK as sitk

def load_array(path: str) -> np.ndarray:
    ext = os.path.splitext(path)[1].lower()
    if ext != '.nrrd':
        raise ValueError(f"Unsupported file type for this notebook (expected .nrrd): {path}")
    image = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(image).astype(np.float32)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D array, got shape {arr.shape} for {path}")
    return arr

def validate_same_shape(arrs: List[np.ndarray]) -> Tuple[int, int, int]:
    shapes = [a.shape for a in arrs]
    if len(set(shapes)) != 1:
        raise ValueError(f"All arrays must have the same shape, got: {shapes}")
    return arrs[0].shape

In [4]:
# Combine and export to CSV (skip all-zero rows) using pandas
import pandas as pd
import os
from pathlib import Path 

def nrrd2csv(input_files: List[str], mask: np.ndarray, output_csv: str):
  arrays = [load_array(p)[mask == 1] for p in input_files]
  validate_same_shape(arrays)
  N = len(arrays)

  # Stack as (N, X, Y, Z) then reshape to (voxels, N)
  stacked = np.stack(arrays, axis=0)  # (N, D, H, W)
  vox_mat = stacked.reshape(N, -1).T       # (D*H*W, N)


  # Create DataFrame and write CSV
  # Column names derived from final token before extension in filename
  # Example: original_firstorder_10Percentile.nrrd -> 10Percentile
  base_names = [os.path.splitext(os.path.basename(p))[0] for p in input_files]
  cols = [bn.split('_')[-1] if '_' in bn else bn for bn in base_names]
  df = pd.DataFrame(vox_mat, columns=cols)

  Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
  df.to_csv(output_csv, index=False)
  print(f"Saved CSV to: {output_csv} with columns: {cols}")

In [5]:
import radiomics
import numpy as np
import SimpleITK as sitk
import radiomics.featureextractor
import os
import six
from os import path
import pandas as pd

def restoreFeatureMapToReference(featureMap, referenceImage):
  """Paste a cropped PyRadiomics map into the full reference image grid."""
  if featureMap.GetDimension() != referenceImage.GetDimension():
    raise ValueError("Feature map và ảnh tham chiếu phải cùng số chiều")

  destinationIndex = referenceImage.TransformPhysicalPointToIndex(
    featureMap.GetOrigin()
  )
  output = sitk.Image(
    referenceImage.GetSize(),
    featureMap.GetPixelID(),
  )
  output.CopyInformation(referenceImage)

  output = sitk.Paste(
    output,
    featureMap,
    featureMap.GetSize(),
    sourceIndex=[0] * featureMap.GetDimension(),
    destinationIndex=destinationIndex,
  )
  return output

def featureExtractor(fileId):
  imagePath = f'./dataset/BraTS2021_Training_Data/{fileId}/{fileId}_flair.nii.gz'
  image = sitk.ReadImage(imagePath)
  maskPath = f'./dataset/BraTS2021_Training_Data/{fileId}/{fileId}_kernel{kernel}_{typeOfVoxel}.nii.gz'
  mask = sitk.ReadImage(maskPath)

  settings = {}
  settings['kernelRadius'] = kernel
  settings['maskedKernel'] = False
  settings['voxelBatch'] = 1000
  extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(**settings)
  extractor.disableAllFeatures()
  extractor.enableFeatureClassByName(className)

  featureMap = extractor.execute(image, mask, voxelBased=True)

  for featureName, featureValue in six.iteritems(featureMap):
    if isinstance(featureValue, sitk.Image):
      fullSizeFeatureMap = restoreFeatureMapToReference(featureValue, mask)
      patientFolder = f'./dataset/10p/{className}/kernel{kernel}/{typeOfVoxel}/{fileId}'
      if path.exists(patientFolder) == False:
        os.makedirs(patientFolder, exist_ok=True)
      sitk.WriteImage(fullSizeFeatureMap, f'{patientFolder}/{featureName}.nrrd')
      print(
        f'Computed {featureName}, stored as "{patientFolder}/{featureName}.nrrd"'
      )
    # else:
    #   print(f'{featureName}: {featureValue}')
  # convert nrrd to csv
  patientPath = patientFolder
  featureFiles = list(filter(lambda f: f.endswith('.nrrd'), os.listdir(patientPath)))
  featureFiles.sort()
  featureFiles = [f"{patientPath}/{featureFile}" for featureFile in featureFiles]
  outputCsvPath = f"{patientPath}/nrrd2csv.csv"
  if os.path.exists(outputCsvPath):
    print(f"CSV already exists for {patientFolder}, skipping.")
    for featureFile in featureFiles:
      os.remove(featureFile)
  else:
    maskArray = sitk.GetArrayFromImage(mask).astype(np.float32)
    nrrd2csv(featureFiles, maskArray, outputCsvPath)
    for featureFile in featureFiles:
      os.remove(featureFile)

In [6]:
# main
monitorFilePath = f"./dataset/10p/{className}/kernel{kernel}/{typeOfVoxel}.monitor.csv"
monitor = pd.read_csv(monitorFilePath, index_col='no')
for i, row in monitor.iterrows():
  if row['done'] != 0:
    continue
  patientId = row['file']
  print('Starting %s' % (patientId))
  featureExtractor(patientId)
  monitor.at[i, 'done'] = 1
  monitor.to_csv(monitorFilePath)

Starting BraTS2021_00000


Computed original_gldm_DependenceEntropy, stored as "./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00000/original_gldm_DependenceEntropy.nrrd"


Computed original_gldm_DependenceNonUniformity, stored as "./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00000/original_gldm_DependenceNonUniformity.nrrd"


Computed original_gldm_DependenceNonUniformityNormalized, stored as "./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00000/original_gldm_DependenceNonUniformityNormalized.nrrd"


Computed original_gldm_DependenceVariance, stored as "./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00000/original_gldm_DependenceVariance.nrrd"


...

Computed original_gldm_LowGrayLevelEmphasis, stored as "./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00014/original_gldm_LowGrayLevelEmphasis.nrrd"


Computed original_gldm_SmallDependenceEmphasis, stored as "./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00014/original_gldm_SmallDependenceEmphasis.nrrd"


Computed original_gldm_SmallDependenceHighGrayLevelEmphasis, stored as "./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00014/original_gldm_SmallDependenceHighGrayLevelEmphasis.nrrd"


Computed original_gldm_SmallDependenceLowGrayLevelEmphasis, stored as "./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00014/original_gldm_SmallDependenceLowGrayLevelEmphasis.nrrd"


Saved CSV to: ./dataset/10p/gldm/kernel5/non_tumor/BraTS2021_00014/nrrd2csv.csv with columns: ['DependenceEntropy', 'DependenceNonUniformity', 'DependenceNonUniformityNormalized', 'DependenceVariance', 'GrayLevelNonUniformity', 'GrayLevelVariance', 'HighGrayLevelEmphasis', 'LargeDependenceEmphasis', 'LargeDependenceHighGrayLevelEmphasis', 'LargeDependenceLowGrayLevelEmphasis', 'LowGrayLevelEmphasis', 'SmallDependenceEmphasis', 'SmallDependenceHighGrayLevelEmphasis', 'SmallDependenceLowGrayLevelEmphasis']
